In [ ]:
# CRISPRAidit: Prediction and SHAP Explainability

This notebook performs prediction and SHAP-based explainability using the CRISPRAidit model. The workflow includes:

- Feature extraction using one-hot, mismatch, PAM, and GC features
- Training an MLPRegressor
- SHAP-based feature importance explanation
- Saving predictions and SHAP values to Excel


In [ ]:
# Setup and Imports 

import os
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler
import shap
import matplotlib.pyplot as plt


In [ ]:
#load the dataset

## Load the Input Dataset

Ensure the Excel file has the required columns like 'target sequence', 'off-target sequence', and 'CRISPRAidit Score'.
data_path = r'C:\\Users\\faiza\\Downloads\\All excel files training testing\\Data sets for all tools with Results\\63nt CRISPRAidit scores data.xlsx'
data = pd.read_excel(data_path, engine='openpyxl')

# If 'off-target sequence' is missing, generate it by reversing 'target sequence'
if 'off-target sequence' not in data.columns:
    data['off-target sequence'] = data['target sequence'].apply(lambda x: x[::-1])

print(" Dataset loaded with shape:", data.shape)



In [ ]:
# Feature Extraction Functions

# One-hot encoding of 23-nt sequence
def one_hot_encode(seq, length=23):
    mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    encoded = np.zeros((length, 4))
    for i, char in enumerate(seq[:length]):
        if char in mapping:
            encoded[i, mapping[char]] = 1
    return encoded.flatten()

# Mismatch features and substitution pairs (12 types × 23 positions)
def compute_mismatch_features(seq1, seq2, length=23):
    mismatch_flags = np.zeros(length)
    nucleotide_pairs = np.zeros((length, 12))
    mismatch_map = {
        ('A', 'C'): 0, ('A', 'G'): 1, ('A', 'T'): 2,
        ('C', 'A'): 3, ('C', 'G'): 4, ('C', 'T'): 5,
        ('G', 'A'): 6, ('G', 'C'): 7, ('G', 'T'): 8,
        ('T', 'A'): 9, ('T', 'C'): 10, ('T', 'G'): 11
    }
    for i in range(length):
        if i < len(seq1) and i < len(seq2) and seq1[i] != seq2[i]:
            mismatch_flags[i] = 1
            pair = (seq1[i], seq2[i])
            if pair in mismatch_map:
                nucleotide_pairs[i, mismatch_map[pair]] = 1
    return np.concatenate([mismatch_flags, nucleotide_pairs.flatten()])


In [ ]:
# Additional Feature Functions

# PAM encoding using one-hot scheme
def pam_encoding(pam_seq):
    pam_dict = {'GG': [1, 0, 0, 0, 0, 0, 0, 0], 
                'AG': [0, 1, 0, 0, 0, 0, 0, 0],
                'GT': [0, 0, 1, 0, 0, 0, 0, 0],
                'GC': [0, 0, 0, 1, 0, 0, 0, 0],
                'GA': [0, 0, 0, 0, 1, 0, 0, 0],
                'TG': [0, 0, 0, 0, 0, 1, 0, 0],
                'CG': [0, 0, 0, 0, 0, 0, 1, 0],
                'other': [0, 0, 0, 0, 0, 0, 0, 1]}
    return pam_dict.get(pam_seq, pam_dict['other'])

# GC content feature
def gc_content(seq):
    return (seq.count('G') + seq.count('C')) / len(seq)


In [ ]:
# Extract All Features

def extract_all_features(data):
    features_list = []
    for idx, row in data.iterrows():
        try:
            seq_features = one_hot_encode(row['target sequence'], length=23)
            mismatch_features = compute_mismatch_features(row['target sequence'], row['off-target sequence'], length=23)
            pam_features = np.array(pam_encoding(row.get('PAM', 'GG')))
            gc_feature = np.array([gc_content(row['target sequence'])])
            placeholder_features = np.zeros(473 - (len(seq_features) + len(mismatch_features) + len(pam_features) + len(gc_feature)))
            combined_features = np.concatenate([seq_features, mismatch_features, pam_features, gc_feature, placeholder_features])
            if len(combined_features) != 473:
                raise ValueError(f"Unexpected feature length: {len(combined_features)}")
            features_list.append(combined_features)
        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            continue
    if not features_list:
        raise ValueError("No valid features extracted.")
    return np.stack(features_list)


In [ ]:
# Feature Names and Target

# Generate full feature names
sequence_feature_names = [f"{nucleotide}{i:02d}" for i in range(23) for nucleotide in "ACGT"]
mismatch_feature_names = ([f"M{i:02d}" for i in range(23)] +
                          [f"{pair}{i:02d}" for i in range(23) for pair in ["AC", "AG", "AT", "CA", "CG", "CT", "GA", "GC", "GT", "TA", "TC", "TG"]])
pam_feature_names = ["PAM_GG", "PAM_AG", "PAM_GT", "PAM_GC", "PAM_GA", "PAM_TG", "PAM_CG", "PAM_other"]
gc_feature_name = ["GC_Content"]
placeholder_feature_names = [f"Placeholder_{i}" for i in range(473 - (len(sequence_feature_names) + len(mismatch_feature_names) + len(pam_feature_names) + len(gc_feature_name)))]

all_feature_names = sequence_feature_names + mismatch_feature_names + pam_feature_names + gc_feature_name + placeholder_feature_names

# Extract features and target
features = extract_all_features(data)
target = data['CRISPRAidit Score']


In [ ]:
# Normalize Features and applied  train Model

scaler = MinMaxScaler()
features_scaled = scaler.fit_transform(features)

# Paths for dataset
data_path = r'C:\\Users\\faiza\\Downloads\\All excel files training testing\\Data sets for all tools with Results\\63nt CRISPRAidit scores data.xlsx'
model_save_path = r'C:\\Users\\faiza\\Downloads\\crispraidit_model.pkl'  # Added model save path
scaler_save_path = r'C:\\Users\\faiza\\Downloads\\crispraidit_scaler.pkl'  # Added scaler save path

# Predict scores
predictions = model.predict(features_scaled)
data['Predictions'] = predictions


In [ ]:
# SHAP Explainability 

print("Calculating SHAP values...")
required_max_evals = 2 * features_scaled.shape[1] + 1
explainer = shap.Explainer(model.predict, features_scaled)
shap_values = explainer(features_scaled, max_evals=required_max_evals)

base_value_unscaled = predictions.mean()


In [ ]:
#extracting  SHAP values from the dataset sequences 

shap_columns = [f"SHAP_Value_{name}" for name in all_feature_names]
shap_df = pd.DataFrame(shap_values.values, columns=shap_columns)
feature_df = pd.DataFrame(features, columns=all_feature_names)
data = pd.concat([data, feature_df, shap_df], axis=1)


In [ ]:
 # Save output to excel file 

output_path = r'C:\\Users\\faiza\\Downloads\\predictions_shap_values_with_all_features.xlsx'
data.to_excel(output_path, index=False)
print(" Predictions and SHAP values saved to:", output_path)


In [ ]:
#  Generate SHAP Waterfall Plots 

for idx in range(len(data)):
    shap_values_object = shap.Explanation(
        values=shap_values[idx].values,
        base_values=base_value_unscaled,
        data=features_scaled[idx],
        feature_names=all_feature_names
    )
    plt.figure(figsize=(10, 6))
    shap.plots.waterfall(shap_values_object, max_display=10, show=True)
    plt.title(f"SHAP Waterfall Plot for Row {idx}")
    plt.savefig(f'C:\\Users\\faiza\\Downloads\\shap_waterfall_plot_row_{idx}.png')
    plt.close()

print("SHAP waterfall plots saved.")


In [ ]:
# Finall code
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler
import shap
import matplotlib.pyplot as plt
import joblib  # Added for model saving

# Paths for dataset
data_path = r'C:\\Users\\faiza\\Downloads\\All excel files training testing\\Data sets for all tools with Results\\63nt CRISPRAidit scores data.xlsx'
model_save_path = r'C:\\Users\\faiza\\Downloads\\crispraidit_model.pkl'  # Added model save path
scaler_save_path = r'C:\\Users\\faiza\\Downloads\\crispraidit_scaler.pkl'  # Added scaler save path

# Load dataset
data = pd.read_excel(data_path, engine='openpyxl')

# Ensure the 'off-target sequence' column exists
if 'off-target sequence' not in data.columns:
    data['off-target sequence'] = data['target sequence'].apply(lambda x: x[::-1])

# Feature extraction functions (unchanged)
def one_hot_encode(seq, length=23):
    mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    encoded = np.zeros((length, 4))
    for i, char in enumerate(seq[:length]):
        if char in mapping:
            encoded[i, mapping[char]] = 1
    return encoded.flatten()

def compute_mismatch_features(seq1, seq2, length=23):
    mismatch_flags = np.zeros(length)
    nucleotide_pairs = np.zeros((length, 12))
    mismatch_map = {
        ('A', 'C'): 0, ('A', 'G'): 1, ('A', 'T'): 2,
        ('C', 'A'): 3, ('C', 'G'): 4, ('C', 'T'): 5,
        ('G', 'A'): 6, ('G', 'C'): 7, ('G', 'T'): 8,
        ('T', 'A'): 9, ('T', 'C'): 10, ('T', 'G'): 11
    }
    for i in range(length):
        if i < len(seq1) and i < len(seq2) and seq1[i] != seq2[i]:
            mismatch_flags[i] = 1
            pair = (seq1[i], seq2[i])
            if pair in mismatch_map:
                nucleotide_pairs[i, mismatch_map[pair]] = 1
    return np.concatenate([mismatch_flags, nucleotide_pairs.flatten()])

def pam_encoding(pam_seq):
    pam_dict = {'GG': [1, 0, 0, 0, 0, 0, 0, 0], 
                'AG': [0, 1, 0, 0, 0, 0, 0, 0],
                'GT': [0, 0, 1, 0, 0, 0, 0, 0],
                'GC': [0, 0, 0, 1, 0, 0, 0, 0],
                'GA': [0, 0, 0, 0, 1, 0, 0, 0],
                'TG': [0, 0, 0, 0, 0, 1, 0, 0],
                'CG': [0, 0, 0, 0, 0, 0, 1, 0],
                'other': [0, 0, 0, 0, 0, 0, 0, 1]}
    return pam_dict.get(pam_seq, pam_dict['other'])

def gc_content(seq):
    return (seq.count('G') + seq.count('C')) / len(seq)

def extract_all_features(data):
    features_list = []
    for idx, row in data.iterrows():
        try:
            seq_features = one_hot_encode(row['target sequence'], length=23)
            mismatch_features = compute_mismatch_features(row['target sequence'], row['off-target sequence'], length=23)
            pam_features = np.array(pam_encoding(row.get('PAM', 'GG')))
            gc_feature = np.array([gc_content(row['target sequence'])])

            placeholder_features = np.zeros(473 - (len(seq_features) + len(mismatch_features) + len(pam_features) + len(gc_feature)))

            combined_features = np.concatenate([seq_features, mismatch_features, pam_features, gc_feature, placeholder_features])

            if len(combined_features) != 473:
                raise ValueError(f"Unexpected feature length: {len(combined_features)}")

            features_list.append(combined_features)
        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            continue

    if not features_list:
        raise ValueError("No valid features extracted.")

    return np.stack(features_list)

# Generate feature names (unchanged)
sequence_feature_names = [f"{nucleotide}{i:02d}" for i in range(23) for nucleotide in "ACGT"]
mismatch_feature_names = ([f"M{i:02d}" for i in range(23)] +
                          [f"{pair}{i:02d}" for i in range(23) for pair in ["AC", "AG", "AT", "CA", "CG", "CT", "GA", "GC", "GT", "TA", "TC", "TG"]])
pam_feature_names = ["PAM_GG", "PAM_AG", "PAM_GT", "PAM_GC", "PAM_GA", "PAM_TG", "PAM_CG", "PAM_other"]
gc_feature_name = ["GC_Content"]
placeholder_feature_names = [f"Placeholder_{i}" for i in range(473 - (len(sequence_feature_names) + len(mismatch_feature_names) + len(pam_feature_names) + len(gc_feature_name)))]

all_feature_names = (sequence_feature_names + mismatch_feature_names + pam_feature_names + gc_feature_name + placeholder_feature_names)

# Extract features and target
features = extract_all_features(data)
target = data['CRISPRAidit Score']

# Normalize features
scaler = MinMaxScaler()
features_scaled = scaler.fit_transform(features)

# Train and save the model
model = MLPRegressor(hidden_layer_sizes=(550, 490, 50, 50, 170), max_iter=330, alpha=0.008813467753305038, random_state=42)
model.fit(features_scaled, target)

# Save the trained model and scaler
joblib.dump(model, model_save_path)
joblib.dump(scaler, scaler_save_path)
print(f"Model saved to: {model_save_path}")
print(f"Scaler saved to: {scaler_save_path}")

# Generate predictions from sequences
data['Predictions'] = model.predict(features_scaled)

# SHAP explanation and saving results (unchanged)
print("Calculating SHAP values...")
explainer = shap.Explainer(model.predict, features_scaled)
shap_values = explainer(features_scaled, max_evals=2 * features_scaled.shape[1] + 1)

shap_columns = [f"SHAP_Value_{name}" for name in all_feature_names]
shap_df = pd.DataFrame(shap_values.values, columns=shap_columns)
feature_df = pd.DataFrame(features, columns=all_feature_names)

data = pd.concat([data, feature_df, shap_df], axis=1)

output_path = r'C:\\Users\\faiza\\Downloads\\predictions_shap_values_with_all_features.xlsx'
data.to_excel(output_path, index=False)
print("Results saved to:", output_path)

# Generate SHAP plot (unchanged)
idx = 0
shap_values_object = shap.Explanation(
    values=shap_values[idx].values,
    base_values=data['Predictions'].mean(),
    data=features_scaled[idx],
    feature_names=all_feature_names
)
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_values_object, max_display=10, show=True)
plt.title(f"SHAP Waterfall Plot for Row {idx}")
plt.savefig(f'C:\\Users\\faiza\\Downloads\\shap_waterfall_plot_row_{idx}.png')
plt.close()

print("Process completed successfully.")